# OpenSDI Preliminary EDA
## Mad-Eye-Moody-Lens — Dataset Analysis

**Contribution:** preliminary characterization of OpenSDI for modern-generator and architecture-OOD analysis.

This notebook:
- loads OpenSDI using its native Hugging Face dataset interface
- samples 100 real + 100 AI-generated records per generator
- checks class/generator/family coverage
- checks mask availability and duplicate keys
- creates plots and exports the cleaned metadata

**Target:** 1,000 records.


In [ ]:
# 1. Environment
import os, json, sys
import pandas as pd
import matplotlib.pyplot as plt

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("Environment ready.")


In [ ]:
# 2. Dataset configuration
from datasets import load_dataset, Image as HFImage

DATASET_ID = "nebula/OpenSDI_test"

GENERATORS = {
    "sd15": ("SD1.5", "UNet diffusion"),
    "sd2": ("SD2.1", "UNet diffusion"),
    "sdxl": ("SDXL", "UNet diffusion"),
    "sd3": ("SD3", "DiT / Flow Matching"),
    "flux": ("Flux.1", "DiT / Flow Matching"),
}

N_PER_LABEL = 100

print("Dataset:", DATASET_ID)
print("Target:", len(GENERATORS) * 2 * N_PER_LABEL, "records")


## 3. Balanced metadata collection

The OpenSDI test dataset exposes the fields `key`, `image`, `mask`, and `label`. The `label` field directly identifies `real` and `fake`.

Images are not decoded or saved during collection. This keeps the preliminary EDA lightweight and reproducible.


In [ ]:
# 3. Collect 100 real + 100 fake from each generator
records = []

for split, (generator_name, family) in GENERATORS.items():
    print(f"\n===== {generator_name} =====")

    ds = load_dataset(DATASET_ID, split=split, streaming=True)

    try:
        ds = ds.cast_column("image", HFImage(decode=False))
        ds = ds.cast_column("mask", HFImage(decode=False))
    except Exception:
        pass

    counts = {"real": 0, "fake": 0}

    for row in ds:
        label = row["label"]

        if isinstance(label, str):
            kind = label
        else:
            kind = "real" if int(label) == 0 else "fake"

        if counts[kind] >= N_PER_LABEL:
            if counts["real"] >= N_PER_LABEL and counts["fake"] >= N_PER_LABEL:
                break
            continue

        key = str(row["key"])

        records.append({
            "sample_id": len(records),
            "key": key,
            "generator": split,
            "generator_name": generator_name,
            "generator_family": family,
            "label": 1 if kind == "fake" else 0,
            "label_name": kind,
            "mask_present": row["mask"] is not None
        })

        counts[kind] += 1

    print("Real:", counts["real"], "| Fake:", counts["fake"])

df = pd.DataFrame(records)

print("\nTOTAL RECORDS:", len(df))
assert len(df) == 1000, f"Expected 1000 records, got {len(df)}"

print("\nBalanced sample:")
print(pd.crosstab(df["generator_name"], df["label_name"]))


## 4. Dataset inventory and quality checks


In [ ]:
# 4. Inventory
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique keys:", df["key"].nunique())
print("Duplicate keys:", df["key"].duplicated().sum())
print("Real:", (df["label"] == 0).sum())
print("AI-generated:", (df["label"] == 1).sum())

print("\nGenerator families:")
print(df["generator_family"].value_counts())

print("\nMask availability:")
print(df["mask_present"].value_counts())


In [ ]:
# 5. Real vs AI
counts = df["label_name"].value_counts()

plt.figure(figsize=(7,4))
counts.plot(kind="bar")
plt.title("OpenSDI Sample: Real vs AI-Generated")
plt.xlabel("Class")
plt.ylabel("Number of Records")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("dataset_analysis/figures/class_balance.png", dpi=160)
plt.show()


In [ ]:
# 6. Generator coverage
order = ["SD1.5", "SD2.1", "SDXL", "SD3", "Flux.1"]
g = df["generator_name"].value_counts().reindex(order)

plt.figure(figsize=(9,4))
g.plot(kind="bar")
plt.title("OpenSDI Sample by Generator")
plt.xlabel("Generator")
plt.ylabel("Number of Records")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("dataset_analysis/figures/generator_coverage.png", dpi=160)
plt.show()


In [ ]:
# 7. Generator x class
gc = pd.crosstab(df["generator_name"], df["label_name"]).reindex(order)

print(gc)

gc.plot(kind="bar", figsize=(9,5))
plt.title("Real vs AI-Generated Across Generators")
plt.xlabel("Generator")
plt.ylabel("Number of Records")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("dataset_analysis/figures/generator_class_balance.png", dpi=160)
plt.show()


In [ ]:
# 8. Architecture family
fc = df["generator_family"].value_counts()

plt.figure(figsize=(8,4))
fc.plot(kind="bar")
plt.title("OpenSDI Sample by Generator Family")
plt.xlabel("Generator Family")
plt.ylabel("Number of Records")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("dataset_analysis/figures/generator_family.png", dpi=160)
plt.show()


In [ ]:
# 9. Mask availability
mc = df["mask_present"].value_counts()

plt.figure(figsize=(6,4))
mc.plot(kind="bar")
plt.title("Mask Availability in Sample")
plt.xlabel("Mask Present")
plt.ylabel("Number of Records")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("dataset_analysis/figures/mask_availability.png", dpi=160)
plt.show()


## 5. Preliminary interpretation

### Dataset suitability

The selected sample provides balanced coverage of real and AI-generated records across five generators. It includes both older UNet diffusion generators and newer SD3/Flux.1 generators grouped under a project-level DiT/Flow-Matching family.

This makes OpenSDI relevant to the project's discussion of modern-generator and architecture-OOD coverage.

### Quality checks

The analysis checks exact duplicate keys and records mask availability. These are metadata-level checks and should not be interpreted as proof that there are no semantic or near-duplicate images.

### Limitation

This is a **dataset characterization / preliminary EDA**, not detector performance evaluation. No classifier is trained here and no generalization accuracy is claimed.


In [ ]:
# 10. Export cleaned metadata and summary
os.makedirs("dataset_analysis/data", exist_ok=True)

csv_path = "dataset_analysis/data/opensdi_cleaned.csv"
json_path = "dataset_analysis/data/opensdi_summary.json"

df.to_csv(csv_path, index=False)

summary = {
    "dataset": DATASET_ID,
    "sample_size": int(len(df)),
    "real_images": int((df["label"] == 0).sum()),
    "ai_generated_images": int((df["label"] == 1).sum()),
    "generators": order,
    "generator_family_counts": df["generator_family"].value_counts().to_dict(),
    "duplicate_keys": int(df["key"].duplicated().sum()),
    "mask_present": int(df["mask_present"].sum()),
    "sampling_rule": "100 real + 100 fake per generator"
}

with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Created:")
print(csv_path)
print(json_path)


In [ ]:
# 11. Package all outputs for submission
import shutil

package = "dataset_analysis_submission"
shutil.rmtree(package, ignore_errors=True)
shutil.copytree("dataset_analysis", package)

zip_name = "OpenSDI_dataset_analysis_submission.zip"
shutil.make_archive("OpenSDI_dataset_analysis_submission", "zip", package)

print("Created:", zip_name)

try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print("Download manually from the Colab Files panel.")


# Final output structure

```text
dataset_analysis/
├── data/
│   ├── opensdi_cleaned.csv
│   └── opensdi_summary.json
├── figures/
│   ├── class_balance.png
│   ├── generator_coverage.png
│   ├── generator_class_balance.png
│   ├── generator_family.png
│   └── mask_availability.png
└── notebook/
    └── OpenSDI_Preliminary_EDA.ipynb
```

**Note:** Do not commit raw OpenSDI images.
